In [ ]:
# 1) Importamos las librerías con sus alias convencionales
import matplotlib.pyplot as plt  # gráficos base (Seaborn se apoya en él)
import numpy as np  # cálculo numérico y condiciones vectorizadas
import pandas as pd  # tablas (DataFrame)
import seaborn as sns  # gráficos estadísticos de alto nivel
from scipy import stats  # pruebas estadísticas (Parte 6)

from flows.common.paths import BRONZE, BRONZE_LABELED_NAME, SILVER, SILVER_NAME

# 2) Estilo global: todos los gráficos heredan esta configuración
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)   # tamaño por defecto (ancho, alto) en pulgadas
plt.rcParams['axes.titlesize'] = 13

# 3) Cargamos Bronze (nunca la URL: la ingesta ya la guardó)
df = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(exclude='number').describe()

In [ ]:
(df['Age'] < 16).groupby(df['Pclass']).mean()

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Pasajeros por género'); plt.show()

In [ ]:
sns.countplot(data=df, x='Sex', hue='Pclass')
plt.title('Género por clase'); plt.show()

In [ ]:
pd.crosstab(df['Sex'], df['Pclass'], normalize='columns').round(3)

In [ ]:
df.describe()
df.select_dtypes(exclude='number').describe()

In [ ]:
print(df.isnull().sum())
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Mapa de valores faltantes (amarillo = falta el dato)')
plt.show()

In [ ]:
df = pd.read_parquet(SILVER / SILVER_NAME)
df.shape

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Pasajeros por género'); plt.show()

sns.countplot(data=df, x='Sex', hue='Pclass')
plt.title('Género por clase'); plt.show()

pd.crosstab(df['Sex'], df['Pclass'], normalize='columns').round(3)

In [ ]:
sns.histplot(df['Age'].dropna(), bins=30)
plt.title('Distribución de edades'); plt.xlabel('Edad (años)'); plt.ylabel('Pasajeros'); plt.show()
df['Age'].mean()
df['Age'].median(), df['Age'].min(), df['Age'].max()

In [ ]:
df['Person'] = np.where(df['Age'] < 16, 'child', df['Sex'])
df['Person'].value_counts()

In [ ]:
pd.crosstab(df['Pclass'], df['Person'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
sns.kdeplot(data=df, x='Age', hue='Sex', fill=True, common_norm=False, ax=axes[0])
sns.kdeplot(data=df, x='Age', hue='Person', fill=True, common_norm=False, ax=axes[1])
sns.kdeplot(data=df, x='Age', hue='Pclass', fill=True, common_norm=False, palette='viridis', ax=axes[2])
axes[0].set_title('Edad por género')
axes[1].set_title('Edad por tipo de persona')
axes[2].set_title('Edad por clase')
plt.tight_layout(); plt.show()

In [ ]:
cabinas = df['Cabin'].dropna()
cubierta = cabinas.str[0]
cubierta.value_counts() 

In [ ]:
orden = sorted(cubierta.unique())
sns.countplot(x=cubierta, order=orden)
plt.title('Pasajeros con cabina por cubierta'); plt.xlabel('Cubierta'); plt.ylabel('Pasajeros'); plt.show()

# La cubierta 'T' tiene un solo registro (anómalo: el barco no tenía cubierta T de pasajeros normales)
cubierta = cubierta[cubierta != 'T']
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Pasajeros con cabina por cubierta (sin anómalos)'); plt.show()


In [ ]:
sns.countplot(data=df, x='Embarked', hue='Pclass', order=['S', 'C', 'Q'])
plt.title('Puerto de embarque por clase'); plt.show()
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(3)


In [ ]:
df['Familia'] = df['SibSp'] + df['Parch']
df['Alone'] = np.where(df['Familia'] > 0, 'with Family', 'Alone')
df['Alone'].value_counts(normalize=True)

In [ ]:
sns.countplot(data=df, x='Alone')
plt.title('Pasajeros solos vs. con familia'); plt.ylabel('Pasajeros'); plt.show()

In [ ]:
df['Survivor'] = df['Survived'].map({0: 'No', 1: 'Yes'})
sns.countplot(data=df, x='Survivor')
plt.title('Supervivientes vs. no supervivientes'); plt.ylabel('Pasajeros'); plt.show()
df['Survived'].mean() 


In [ ]:
g = sns.catplot(data=df, x='Pclass', y='Survived', kind='point', height=4, aspect=1.4)
g.set_axis_labels('Clase', 'Tasa de supervivencia')
plt.title('Supervivencia por clase'); plt.show()


In [ ]:
g = sns.catplot(data=df, x='Pclass', y='Survived', hue='Person', kind='point',
                hue_order=['child', 'female', 'male'], height=4, aspect=1.5)
g.set_axis_labels('Clase', 'Tasa de supervivencia')
plt.show()

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived')
sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass')
generaciones = [10, 20, 40, 60, 80]
sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass', x_bins=generaciones)
sns.lmplot(data=df, x='Age', y='Survived', hue='Sex', x_bins=generaciones)
plt.show()

In [ ]:
plt.scatter(df['Age'], df['Fare'], alpha=0.5)
plt.xlabel('Edad'); plt.ylabel('Tarifa'); plt.title('Tarifa vs. edad'); plt.show()

sns.scatterplot(data=df, x='Age', y='Fare', hue='Pclass', alpha=0.5, palette='viridis')
plt.title('Tarifa vs. edad por clase'); plt.show()

In [ ]:
df['Embarked'].isnull().sum() 
moda = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(moda)
df['NEmbarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})


In [ ]:
std_antes = df['Age'].std() 
promedios = df.groupby('Sex')['Age'].mean()
df['Age'] = df['Age'].fillna(df['Sex'].map(promedios))
df['Age'].isnull().sum()  

In [ ]:
df['NSex'] = df['Sex'].map({'female': 0, 'male': 1})

In [ ]:
df['Cabin'] = df['Cabin'].fillna('X')
df['TCabin'] = df['Cabin'].str[0]
df['TCabin'].value_counts() 
mapa = {letra: i for i, letra in enumerate(sorted(df['TCabin'].unique()))}
df['NCabin'] = df['TCabin'].map(mapa)

In [ ]:
df['TTitulo'] = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
df['TTitulo'].value_counts()  
pd.crosstab(df['TTitulo'], df['Sex'])

In [ ]:
reemplazos = {'Master': 'Mr',
              'the Countess': 'Mrs', 'Dona': 'Mrs', 'Mme': 'Mrs',
              'Lady': 'Miss', 'Mlle': 'Miss', 'Ms': 'Miss'}
df['TTitulo'] = df['TTitulo'].replace(reemplazos)
df['TTitulo'] = df['TTitulo'].where(df['TTitulo'].isin(['Mr', 'Mrs', 'Miss']), 'Otro')
df['TTitulo'].value_counts()
df['NTitulo'] = df['TTitulo'].map({'Mr': 0, 'Mrs': 1, 'Miss': 2, 'Otro': 3})

In [ ]:
cols = ['Survived', 'Pclass', 'NSex', 'Age', 'SibSp', 'Parch', 'Fare', 'NEmbarked', 'NCabin', 'NTitulo']
corr = df[cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlación'); plt.show()
corr['Survived'].sort_values()


In [ ]:
df.groupby('Survived')['Age'].describe()
df.groupby('Sex')['Survived'].agg(['mean', 'sum', 'count'])
df.groupby('Pclass')['Survived'].agg(['mean', 'sum', 'count'])
df.groupby('Alone')['Survived'].agg(['mean', 'sum', 'count'])

In [ ]:
tabla = pd.crosstab(df['Sex'], df['Survived'])
chi2, p, gl, esperados = stats.chi2_contingency(tabla)
print(f'chi2 = {chi2:.1f}, p = {p:.2e}, gl = {gl}')
# p del orden de 1e-58  ->  se rechaza H0: Sex y Survived NO son independientes

In [ ]:
tabla_c = pd.crosstab(df['Pclass'], df['Survived'])
chi2_c, p_c, gl_c, esp_c = stats.chi2_contingency(tabla_c)
print(f'chi2 = {chi2_c:.1f}, p = {p_c:.2e}') 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.boxplot(data=df, x='Sex', y='Age', ax=axes[0, 0])
sns.boxplot(data=df, x='Pclass', y='Age', ax=axes[0, 1])
sns.boxplot(data=df, x='Embarked', y='Age', ax=axes[1, 0])
sns.boxplot(data=df, x='Survived', y='Age', ax=axes[1, 1])
axes[0, 0].set_title('Edad por género'); axes[0, 1].set_title('Edad por clase')
axes[1, 0].set_title('Edad por puerto'); axes[1, 1].set_title('Edad por supervivencia')
plt.tight_layout(); plt.show()


In [ ]:
sns.histplot(df['Fare'], kde=True, bins=40)
plt.title('Distribución de la tarifa'); plt.xlabel('Tarifa'); plt.show()
df['Fare'].mean(), df['Fare'].median(), df['Fare'].skew()

In [ ]:
sns.histplot(np.log1p(df['Fare']), kde=True, bins=40)
plt.title('Distribución de log(1 + Fare)'); plt.show()

In [ ]:
df_final = df.drop(columns=['Name', 'Sex', 'Embarked', 'Cabin'])


In [ ]:
display(df_final.isnull().sum())
display(df_final.dtypes)
df_final.info()
df_final.describe()

In [ ]:
df_final['NAlone'] = df_final['Alone'].map({'with Family': 0, 'Alone': 1})
df_final['NPerson'] = df_final['Person'].map({'child': 0, 'female': 1, 'male': 2})

In [ ]:
df.pivot_table(values='Survived', index='Pclass', columns='Sex', aggfunc='mean').round(3)

In [ ]:
df.groupby('Embarked')['Survived'].agg(['mean', 'count'])
df.pivot_table(values='Survived', index='Embarked', columns='Pclass', aggfunc='mean').round(2)

In [ ]:
df['FareQ'] = df.groupby('Pclass')['Fare'].transform(
    lambda s: pd.qcut(s, 4, labels=False, duplicates='drop'))
df.pivot_table(values='Survived', index='FareQ', columns='Pclass', aggfunc='mean').round(2)


In [ ]:
df.groupby('Familia')['Survived'].agg(['mean', 'count'])
sns.pointplot(data=df, x='Familia', y='Survived')
plt.title('Supervivencia según número de familiares a bordo'); plt.show()

In [ ]:
def outliers_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)

for col in ['Fare', 'Age', 'Familia']:
    print(col, outliers_iqr(df[col]).sum())

mask_fare = outliers_iqr(df['Fare'])
df.groupby(mask_fare)['Survived'].agg(['mean', 'count'])
df.nlargest(10, 'Fare')[['Name', 'Pclass', 'Fare', 'Survived']]

tabla_a = pd.crosstab(df['Alone'], df['Survived'])
chi2_a, p_a, gl_a, esp_a = stats.chi2_contingency(tabla_a)
print(f'chi2 = {chi2_a:.1f}, p = {p_a:.2e}')